# House Orientation Analysis

## Overview

This notebook analyzes property orientations by determining which direction each house faces based on spatial relationships between parcels, roads, and addresses.

## Step 1: Package Installation

Install required geospatial and data analysis packages with compatible versions to avoid dependency conflicts.


In [27]:
# # Install packages with compatible versions to avoid dependency conflicts
# %pip install --no-deps geopandas
# %pip install --no-deps "shapely>=2.0"
# %pip install --no-deps pyproj
# %pip install --no-deps rtree
# %pip install --no-deps fiona
# %pip install --no-deps pyogrio
# %pip install --no-deps folium
# # Install numpy with a compatible version
# %pip install "numpy>=1.21,<2.0"
# # Install pandas (should be compatible with the numpy version above)
# %pip install pandas

## Step 2: Package Verification

Test that all required packages can be imported successfully and display their versions for troubleshooting.


In [1]:
# Test imports to verify all packages are working
try:
    import geopandas as gpd
    import shapely
    import pyproj
    import rtree
    import fiona
    import pyogrio
    import folium
    import numpy as np
    import pandas as pd

    print("✓ All packages imported successfully!")
    print(f"GeoPandas version: {gpd.__version__}")
    print(f"Shapely version: {shapely.__version__}")
    print(f"NumPy version: {np.__version__}")
    print(f"Pandas version: {pd.__version__}")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Some packages may need to be installed manually.")

/Users/louistran/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


✓ All packages imported successfully!
GeoPandas version: 1.1.1
Shapely version: 2.1.2
NumPy version: 1.26.4
Pandas version: 2.3.3


## Step 3: Data Path Setup

Verify that the dataset folder and required files exist. This ensures we have access to the cadastral data, road networks, and address information needed for the analysis.


In [2]:
import os, sys

BASE = os.path.abspath(".")
DATA_DIR = os.path.join(BASE, "dataset")

CAD_PATH = os.path.join(DATA_DIR, "cadastre.gpkg")
ROADS_PATH = os.path.join(DATA_DIR, "roads.gpkg")
GNAF_PATH = os.path.join(DATA_DIR, "gnaf_prop.parquet")
# Optional, unused here:
TX_PATH = os.path.join(DATA_DIR, "transactions.parquet")

for p in [DATA_DIR, CAD_PATH, ROADS_PATH, GNAF_PATH]:
    assert os.path.exists(p), f"Missing: {p}"
print("✓ Found dataset folder and required files.")

✓ Found dataset folder and required files.


## Step 4: Core Function Definitions

Define essential functions for spatial analysis:

- **Layer selection**: Automatically find the correct layers in GeoPackage files
- **Geometry handling**: Convert coordinates to proper geometries and handle different CRS
- **Edge extraction**: Extract polygon edges, handling both single polygons and multipolygons
- **Bearing calculation**: Calculate compass bearings between points
- **Direction conversion**: Convert bearings to compass directions (N, NE, E, etc.)


In [3]:
import os, glob, sys

print("CWD:", os.getcwd())
print("dataset exists:", os.path.isdir("./dataset"))
print("dataset contents:", glob.glob("./dataset/*"))

CAD_PATH = "./dataset/cadastre.gpkg"
ROADS_PATH = "./dataset/roads.gpkg"
GNAF_PATH = "./dataset/gnaf_prop.parquet"

for p in [CAD_PATH, ROADS_PATH, GNAF_PATH]:
    print(p, "exists:", os.path.exists(p))

CWD: /Users/louistran/Desktop/task 2
dataset exists: True
dataset contents: ['./dataset/cadastre.gpkg', './dataset/roads.gpkg', './dataset/transactions.parquet', './dataset/gnaf_prop.parquet']
./dataset/cadastre.gpkg exists: True
./dataset/roads.gpkg exists: True
./dataset/gnaf_prop.parquet exists: True


In [4]:
import math, warnings, numpy as np, pandas as pd, geopandas as gpd
from shapely.geometry import Point, LineString, Polygon, MultiPolygon
import fiona

warnings.filterwarnings("ignore", category=UserWarning)


def pick_layer(gpkg_path, hints):
    layers = fiona.listlayers(gpkg_path)
    for h in hints:
        for L in layers:
            if h.lower() in L.lower():
                return L
    return layers[0]


def ensure_geometry_from_lonlat(df):
    lon_cols = [
        c for c in df.columns if c.lower() in ("lon", "lng", "longitude", "x", "long")
    ]
    lat_cols = [c for c in df.columns if c.lower() in ("lat", "latitude", "y")]
    if lon_cols and lat_cols:
        lon, lat = lon_cols[0], lat_cols[0]
        return gpd.GeoDataFrame(
            df, geometry=gpd.points_from_xy(df[lon], df[lat]), crs="EPSG:4326"
        )
    if "geometry" in df.columns:
        return gpd.GeoDataFrame(df, geometry="geometry")
    raise ValueError("No lon/lat or geometry found in gnaf_prop.parquet.")


def to_local_metric_crs(gdf):
    gdf4326 = gdf.to_crs(4326)
    lon = float(gdf4326.geometry.x.mean())
    zone = int((lon + 180) // 6) + 1
    epsg = 32700 + zone if gdf4326.geometry.y.mean() < 0 else 32600 + zone
    return gdf.to_crs(epsg), epsg


def segmentize_polygon_edges(poly):
    """
    Extract edges from a polygon or multipolygon geometry.
    Handles both Polygon and MultiPolygon objects.
    """
    if poly is None or poly.is_empty:
        return []

    edges = []

    # Handle MultiPolygon objects
    if hasattr(poly, "geoms"):  # MultiPolygon
        for geom in poly.geoms:
            if hasattr(geom, "exterior") and geom.exterior is not None:
                coords = list(geom.exterior.coords)
                edges.extend(
                    [
                        LineString([coords[i], coords[i + 1]])
                        for i in range(len(coords) - 1)
                    ]
                )
    # Handle single Polygon objects
    elif hasattr(poly, "exterior") and poly.exterior is not None:
        coords = list(poly.exterior.coords)
        edges = [LineString([coords[i], coords[i + 1]]) for i in range(len(coords) - 1)]

    return edges


def bearing_deg(p_from: Point, p_to: Point) -> float:
    dx, dy = p_to.x - p_from.x, p_to.y - p_from.y
    return (90 - math.degrees(math.atan2(dy, dx))) % 360  # 0°=N, 90°=E


def compass8(b: float) -> str:
    dirs = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
    return dirs[int((b + 22.5) // 45) % 8]

In [5]:
CAD_HINTS = ["parcel", "cad", "lot", "property", "cadastre"]
ROAD_HINTS = ["road", "street", "transport", "centerline", "roads"]

CAD_LAYER = pick_layer(CAD_PATH, CAD_HINTS)
ROAD_LAYER = pick_layer(ROADS_PATH, ROAD_HINTS)
print("Using layers:", CAD_LAYER, "/", ROAD_LAYER)

cad = gpd.read_file(CAD_PATH, layer=CAD_LAYER)  # polygons
roads = gpd.read_file(ROADS_PATH, layer=ROAD_LAYER)  # lines
gnaf_df = pd.read_parquet(GNAF_PATH)

addr_cols = [
    c
    for c in gnaf_df.columns
    if any(
        k in c.lower() for k in ("address", "formatted", "full", "display", "street")
    )
]
addr = ensure_geometry_from_lonlat(gnaf_df)
addr["address_text"] = (
    gnaf_df[addr_cols[0]]
    if addr_cols
    else gnaf_df.get("address", gnaf_df.index.astype(str))
)

Using layers: cadastre / roads


In [6]:
addr_m, epsg = to_local_metric_crs(addr)
cad_m = cad.to_crs(epsg)
roads_m = roads.to_crs(epsg)
print("Working CRS EPSG:", epsg)

Working CRS EPSG: 32756


In [8]:
# Fixed version of the nearest joins with proper data alignment
nearest_parcel = gpd.sjoin_nearest(
    addr_m[["address_text", "geometry"]],
    cad_m[["geometry"]],
    how="left",
    distance_col="dist_to_parcel",
).rename(columns={"index_right": "parcel_idx"})

nearest_parcel = nearest_parcel.join(
    cad_m[["geometry"]], on="parcel_idx", rsuffix="_parcel"
)
nearest_parcel = gpd.GeoDataFrame(nearest_parcel, geometry="geometry")

parcel_centroids = nearest_parcel["geometry_parcel"].centroid
parcel_gdf = gpd.GeoDataFrame(
    nearest_parcel[["address_text"]], geometry=parcel_centroids, crs=epsg
)

parcel_to_road = gpd.sjoin_nearest(
    parcel_gdf, roads_m[["geometry"]], how="left", distance_col="parcel_road_dist"
).rename(columns={"index_right": "road_idx"})

# Merge the road information back to nearest_parcel using the index
# This ensures proper alignment and avoids length mismatch errors
nearest_parcel = nearest_parcel.merge(
    parcel_to_road[["road_idx", "parcel_road_dist"]],
    left_index=True,
    right_index=True,
    how="left",
)

print(f"✓ Successfully created nearest_parcel with {len(nearest_parcel)} rows")
print(f"Columns: {list(nearest_parcel.columns)}")

✓ Successfully created nearest_parcel with 617622 rows
Columns: ['address_text', 'geometry', 'parcel_idx', 'dist_to_parcel', 'geometry_parcel', 'road_idx', 'parcel_road_dist']


test


In [35]:
def frontage_bearing(row):
    poly = row["geometry_parcel"]
    if poly is None or poly.is_empty:
        return np.nan, None
    try:
        road_geom = roads_m.loc[row["road_idx"], "geometry"]
    except Exception:
        road_geom = None

    edges = segmentize_polygon_edges(poly)
    if not edges:
        return np.nan, None

    # Prefer edge whose midpoint is closest to the nearest road
    if (road_geom is not None) and (not road_geom.is_empty):
        best_mid, best_near, best_d = None, None, 1e18
        for seg in edges:
            mid = seg.interpolate(0.5, normalized=True)
            near_pt = road_geom.interpolate(road_geom.project(mid))
            d = mid.distance(near_pt)
            if d < best_d:
                best_mid, best_near, best_d = mid, near_pt, d
        if best_mid is not None and best_near is not None:
            return bearing_deg(best_mid, best_near), "road"

    # Fallback: longest edge direction (proxy if no road hit)
    longest = max(edges, key=lambda s: s.length)
    p0, p1 = list(longest.coords)
    return bearing_deg(Point(p0), Point(p1)), "edge"


res = nearest_parcel.apply(lambda r: frontage_bearing(r), axis=1)
nearest_parcel["bearing_deg"] = [t[0] for t in res]
nearest_parcel["bearing_method"] = [t[1] for t in res]
nearest_parcel["orientation"] = nearest_parcel["bearing_deg"].apply(
    lambda b: compass8(b) if pd.notnull(b) else None
)

KeyboardInterrupt: 

In [9]:
# Create a small test dataset for performance testing
print(f"Original dataset size: {len(nearest_parcel)} rows")

# Take a small sample (first 100 rows) for testing
test_sample = nearest_parcel.head(100).copy()
print(f"Test sample size: {len(test_sample)} rows")

# Check the data structure
print(f"Test sample columns: {list(test_sample.columns)}")
print(f"Sample of parcel_idx values: {test_sample['parcel_idx'].head()}")
print(f"Sample of road_idx values: {test_sample['road_idx'].head()}")

# Check for any missing values in key columns
print(f"\nMissing values in test sample:")
print(f"parcel_idx missing: {test_sample['parcel_idx'].isna().sum()}")
print(f"road_idx missing: {test_sample['road_idx'].isna().sum()}")
print(f"geometry_parcel missing: {test_sample['geometry_parcel'].isna().sum()}")

Original dataset size: 617622 rows
Test sample size: 100 rows
Test sample columns: ['address_text', 'geometry', 'parcel_idx', 'dist_to_parcel', 'geometry_parcel', 'road_idx', 'parcel_road_dist']
Sample of parcel_idx values: 0    1
0    1
0    1
0    1
1    1
Name: parcel_idx, dtype: int64
Sample of road_idx values: 0      3
0      3
0     24
0    172
1      3
Name: road_idx, dtype: int64

Missing values in test sample:
parcel_idx missing: 0
road_idx missing: 0
geometry_parcel missing: 0


In [10]:
OUT_CSV = "property_orientation.csv"
out = nearest_parcel[
    ["address_text", "bearing_deg", "orientation", "parcel_road_dist"]
].copy()
out = out.rename(
    columns={"address_text": "address", "parcel_road_dist": "distance_to_road_m"}
)
out.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV}")
print(out.head(10).to_string(index=False))

KeyError: "['bearing_deg', 'orientation'] not in index"

In [11]:
# Test the frontage_bearing function on a small sample
import time


def frontage_bearing(row):
    """Calculate the bearing of the frontage (edge closest to road)"""
    try:
        poly = row["geometry_parcel"]
        road_idx = row["road_idx"]
        if pd.isna(road_idx):
            return np.nan, "no_road"
        road_geom = roads_m.iloc[int(road_idx)]["geometry"]
    except Exception:
        road_geom = None

    edges = segmentize_polygon_edges(poly)
    if not edges:
        return np.nan, None

    # Prefer edge whose midpoint is closest to the nearest road
    if road_geom is not None:
        midpoints = [e.interpolate(0.5, normalized=True) for e in edges]
        distances = [mp.distance(road_geom) for mp in midpoints]
        best_idx = np.argmin(distances)
        longest = edges[best_idx]
    else:
        # Fallback: use the longest edge
        lengths = [e.length for e in edges]
        longest = edges[np.argmax(lengths)]

    p0, p1 = list(longest.coords)
    return bearing_deg(Point(p0), Point(p1)), "edge"


# Test on small sample
print("Testing frontage_bearing function on small sample...")
start_time = time.time()

# Apply the function to our test sample
test_results = test_sample.apply(lambda r: frontage_bearing(r), axis=1)

end_time = time.time()
processing_time = end_time - start_time

print(f"✓ Processing completed in {processing_time:.2f} seconds")
print(f"✓ Processed {len(test_sample)} properties")
print(
    f"✓ Average time per property: {processing_time/len(test_sample)*1000:.2f} milliseconds"
)

# Check results
test_sample["bearing_deg"] = [t[0] for t in test_results]
test_sample["bearing_method"] = [t[1] for t in test_results]

print(f"\nResults summary:")
print(f"Valid bearings: {test_sample['bearing_deg'].notna().sum()}")
print(f"Invalid bearings: {test_sample['bearing_deg'].isna().sum()}")
print(f"Sample bearings: {test_sample['bearing_deg'].head(10).tolist()}")

Testing frontage_bearing function on small sample...
✓ Processing completed in 0.47 seconds
✓ Processed 100 properties
✓ Average time per property: 4.71 milliseconds

Results summary:
Valid bearings: 100
Invalid bearings: 0
Sample bearings: [324.0462224244404, 324.0462224244404, 248.60232423803114, 248.60232423803114, 324.0462224244404, 248.60232423803114, 248.60232423803114, 248.60232423803114, 324.0462224244404, 324.0462224244404]


In [12]:
# Convert bearings to compass directions and show results
test_sample["orientation"] = test_sample["bearing_deg"].apply(
    lambda x: compass8(x) if pd.notna(x) else "Unknown"
)

print("Test Results - Orientation Distribution:")
print(test_sample["orientation"].value_counts())

print(f"\nSample of results:")
sample_results = test_sample[["address_text", "bearing_deg", "orientation"]].head(10)
print(sample_results)

# Estimate full dataset processing time
estimated_full_time = (processing_time / len(test_sample)) * len(nearest_parcel)
print(f"\nPerformance Estimate:")
print(f"Full dataset processing time: ~{estimated_full_time/60:.1f} minutes")
print(f"Full dataset processing time: ~{estimated_full_time/3600:.1f} hours")

# Ask user if they want to proceed with full dataset
print(f"\nRecommendation:")
if estimated_full_time > 300:  # More than 5 minutes
    print("⚠️  Full dataset processing will take a long time.")
    print("Consider using a larger sample (e.g., 1000-5000 rows) for testing first.")
else:
    print("✓ Processing time looks reasonable. You can proceed with the full dataset.")

Test Results - Orientation Distribution:
orientation
W     44
NW    39
SW    17
Name: count, dtype: int64

Sample of results:
  address_text  bearing_deg orientation
0   NSW2878308   324.046222          NW
0   NSW2878308   324.046222          NW
0   NSW2878308   248.602324           W
0   NSW2878308   248.602324           W
1   NSW2878308   324.046222          NW
1   NSW2878308   248.602324           W
1   NSW2878308   248.602324           W
1   NSW2878308   248.602324           W
2   NSW2878308   324.046222          NW
2   NSW2878308   324.046222          NW

Performance Estimate:
Full dataset processing time: ~48.5 minutes
Full dataset processing time: ~0.8 hours

Recommendation:
⚠️  Full dataset processing will take a long time.
Consider using a larger sample (e.g., 1000-5000 rows) for testing first.


## Step 13: Optimized Full Dataset Processing (Optional)

If the test results look good and performance is acceptable, you can run the orientation calculation on the full dataset.

**Note**: This cell will only run if you're satisfied with the test results above.


In [13]:
# OPTIONAL: Run on full dataset if test results are satisfactory
# Uncomment the lines below to process the full dataset

# print("Processing full dataset...")
# start_time = time.time()

# # Apply to full dataset
# full_results = nearest_parcel.apply(lambda r: frontage_bearing(r), axis=1)
# nearest_parcel["bearing_deg"] = [t[0] for t in full_results]
# nearest_parcel["bearing_method"] = [t[1] for t in full_results]

# end_time = time.time()
# full_processing_time = end_time - start_time

# print(f"✓ Full dataset processing completed in {full_processing_time/60:.1f} minutes")
# print(f"✓ Processed {len(nearest_parcel)} properties")

# # Convert to orientations
# nearest_parcel['orientation'] = nearest_parcel['bearing_deg'].apply(
#     lambda x: compass8(x) if pd.notna(x) else "Unknown"
# )

# print(f"✓ Orientation calculation completed")
# print(f"Orientation distribution:")
# print(nearest_parcel['orientation'].value_counts())

print("Full dataset processing is commented out.")
print("Uncomment the code above to run on the full dataset after testing.")

Full dataset processing is commented out.
Uncomment the code above to run on the full dataset after testing.


In [ ]:
import folium

# If you still have original address points in EPSG:4326:
addr_sample = addr.sample(min(50, len(addr)), random_state=7).to_crs(4326)
m = folium.Map(
    location=[addr_sample.geometry.y.mean(), addr_sample.geometry.x.mean()],
    zoom_start=12,
)
for i, r in addr_sample.iterrows():
    folium.CircleMarker(
        [r.geometry.y, r.geometry.x],
        radius=4,
        tooltip=str(addr.loc[i, "address_text"]),
        fill=True,
    ).add_to(m)
m  # In notebooks, this displays an interactive map